# Comparación de modelos de machine learning para la predicción de cáncer de pulmón

## Importación de librerías y modelos

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import VarianceThreshold
from sklearn.utils import resample
import optuna
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

print("Librerías importadas correctamente")

## Proceso ETL y visualizaciones

### Carga de Dataset

In [ ]:
DATA_PATH = "data/dataset.csv"

df = pd.read_csv(DATA_PATH, sep=";")
print(f"Datos: {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

### Variable objetivo

In [ ]:
target = 'Final_Prediction'
df[target] = df[target].map({'Yes':1, 'No':0})

### Eliminación de valores nulos

In [ ]:
# Rellenar nulos con la moda
if 'Treatment_Access' in df.columns:
    mode_val = df['Treatment_Access'].mode()[0]
    df['Treatment_Access'] = df['Treatment_Access'].fillna(mode_val)

# Rellenar nulos con Unkmown
if 'Mutation_Type' in df.columns:
    df['Mutation_Type'] = df['Mutation_Type'].fillna('Unknown')

### Mapeo para variables ordinales

In [ ]:
ordinal_mappings = {
    'Smoking_Status': {'Non-Smoker': 0, 'Former Smoker': 1, 'Smoker': 2},
    'Air_Pollution_Exposure': {'Low': 0, 'Medium': 1, 'High': 2},
    'Socioeconomic_Status': {'Low': 0, 'Middle': 1, 'High': 2},
    'Healthcare_Access': {'Poor': 0, 'Limited': 1, 'Good': 2},
    'Stage_at_Diagnosis': {'I': 1, 'II': 2, 'III': 3, 'IV': 4},
    'Treatment_Access': {'None': 0, 'Partial': 1, 'Full': 2}
}

# Aplicar mapeo
for col, mapping in ordinal_mappings.items():
  if col in df.columns:
    df[col] = df[col].map(mapping)

### Mapeo para variables binarias

In [ ]:
binary_cols = ['Second_Hand_Smoke', 'Occupation_Exposure', 'Insurance_Coverage',
               'Screening_Availability', 'Clinical_Trial_Access', 'Language_Barrier',
               'Delay_in_Diagnosis', 'Family_History', 'Indoor_Smoke_Exposure',
               'Tobacco_Marketing_Exposure', 'Gender']

# Mapeo
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in binary_cols:
  if col in df.columns:
    df[col] = df[col].map(binary_map)

### Columnas numéricas continuas (escalar) y nominales restantes (one-hot)

In [ ]:
# Columnas numéricas continuas
numeric_cols = ['Age', 'Mortality_Risk', '5_Year_Survival_Probability']
numeric_cols = [col for col in numeric_cols if col in df.columns]

# Columnas nominales restantes
already_handled = set(ordinal_mappings.keys()) | set(binary_cols) | set(numeric_cols) | {target} # Escluir las anteriores
categorical_nominal = [col for col in df.columns if col not in already_handled]

# One-hot
df = pd.get_dummies(df, columns=categorical_nominal, drop_first=True)

df.head()

### Convertir variables booleanas a enteros

In [ ]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Verificación final

In [ ]:
print(f"Total de nulos en el dataset: {df.isnull().sum().sum()}")

In [ ]:
df.info()

In [ ]:
df.describe()

### Correlación con variable objetivo
No existe una correlación fuerte en ninguna variable

In [ ]:
# Calcular correlación con el objetivo y ordenar
corr_target = df.corr()['Final_Prediction'].drop('Final_Prediction').sort_values(ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x=corr_target.values, y=corr_target.index, palette='viridis')
plt.title('Correlación de cada variable con Final_Prediction')
plt.xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

### Heatmap con 15 variables más correlacionadas

In [ ]:
top_vars = corr_target.abs().nlargest(15).index.tolist() + ['Final_Prediction']
plt.figure(figsize=(12, 10))
sns.heatmap(df[top_vars].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Heatmap de las 15 variables más correlacionadas con el objetivo')
plt.tight_layout()
plt.show()

## Ingeniería de características avanzada

In [ ]:
# Copia de trabajo
df_feat = df.copy()
target_col = 'Final_Prediction'

### Agrupar países a continentes

In [ ]:
country_cols = [c for c in df_feat.columns if c.startswith('Country_')]

if country_cols:
  mapeo_continente = {
    'USA':'América', 'Canada':'América', 'Mexico':'América', 'Brazil':'América',
    'Colombia':'América', 'Argentina':'América', 'Peru':'América', 'Chile':'América',
    'UK':'Europa', 'Germany':'Europa', 'France':'Europa', 'Spain':'Europa',
    'Italy':'Europa', 'Russia':'Europa/Asia', 'Turkey':'Asia',
    'China':'Asia', 'Japan':'Asia', 'India':'Asia', 'South Korea':'Asia',
    'Indonesia':'Asia', 'Pakistan':'Asia', 'Bangladesh':'Asia', 'Vietnam':'Asia',
    'Thailand':'Asia', 'Philippines':'Asia', 'Myanmar':'Asia', 'Iran':'Asia',
    'Egypt':'África', 'Nigeria':'África', 'Kenya':'África', 'Ethiopia':'África',
    'Tanzania':'África', 'DR Congo':'África', 'South Africa':'África'
  }

  continente = pd.Series(0, index=df_feat.index)
  for col in country_cols:
    pais = col.replace('Country_', '')
    cont = mapeo_continente.get(pais, 'Otro')
    cod = {'América': 1, 'Europa': 2, 'Asia':3, 'África':4, 'Otro':5}.get(cont, 0)
    continente += df_feat[col] * cod
  for cont in ['América','Europa','Asia','África','Otro']:
    df_feat[f'Continente_{cont}'] = (continente == {'América':1,'Europa':2,'Asia':3,'África':4,'Otro':5}.get(cont,0)).astype(int)
  df_feat.drop(columns=country_cols, inplace=True)
  print(f"Países agrupados en continentes. Tamaño actual: {df_feat.shape}")

### Eliminar columnas con varianza menor a 0.01

In [ ]:
X_temp = df_feat.drop(columns=[target_col])
selector_var = VarianceThreshold(threshold=0.01)
X_high_var = selector_var.fit_transform(X_temp)
cols_keep = X_temp.columns[selector_var.get_support()]
df_feat  =pd.concat([df_feat[cols_keep], df_feat[target_col]], axis=1)
print(f"Columnas con baja varianza elimiandas. Tamaño actual: {df_feat.shape}")

### Importancia con RandomForest y crear interacciones

In [ ]:
X_imp = df_feat.drop(columns=[target_col])
y_imp = df_feat[target_col]
rf_imp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_imp.fit(X_imp, y_imp)
importancias = pd.Series(rf_imp.feature_importances_, index=X_imp.columns)
top10 = importancias.nlargest(10).index.tolist()
print(f"Top 10 características: {top10}")

In [ ]:
# Interacciones multiplicativas 
for i in range(len(top10)):
  for j in range(i+1, len(top10)):
    df_feat[f'{top10[i]}_x_{top10[j]}'] = df_feat[top10[i]] * df_feat[top10[j]]

print(f"Interacciones añadidas: {len(top10)*(len(top10)-1)//2}. Tamaño final: {df_feat.shape}") # 10 * 9 / 2 = 45

# Reemplazar dataset original
df = df_feat.copy()

## Entrenamiento de modelos predictivos

### División entrenamiento/prueba

In [ ]:
X = df.drop(target, axis=1) # Características
y = df[target]              # Variable objetivo

# Entrenamiento/prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Distribución de la variable objetivo
print("Distribución de Final_Prediction:")
print(y.value_counts(normalize=True))

print("\nTop 5 correlaciones con el objetivo:")
corrs = X.corrwith(y).abs().sort_values(ascending=False)
print(corrs.head())


### Escalado y Oversampling

In [ ]:
# Escalado de columnas numéricas (valores 0 a 1)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
# Escalar columnas numéricas identificadas
for col in numeric_cols:
  if col in X_train.columns:
    X_train_scaled[col] = scaler.fit_transform(X_train[[col]]) # Calcula parámetros necesarios y transforma en esos mismos datos. Solo se usa en X_train
    X_test_scaled[col] = scaler.transform(X_test[[col]])       # Utiliza parámetros ya calculados. Solo se usa en X_test

# Aumento de datos con SMOTE al conjunto de entrenamiento (oversampling)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"Tamaño original de entrenamiento: {X_train.shape[0]}")
print(f"Tamaño alterado con SMOTE de entrenamiento: {X_train_resampled.shape[0]}")

### Optimización de muestreo

In [ ]:
SAMPLE_SIZE = 40000 # Tamaño balanceado para buscar hiperparámetros
X_sample, y_sample = resample(X_train_resampled, y_train_resampled,
                              n_samples=SAMPLE_SIZE, stratify=y_train_resampled,
                              random_state=42)

print(f"Tamaño muestra para optimización: {X_sample.shape[0]}")

# Validación cruzada estratificada (realizar 2 entrenamientos y validar con 1)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

### Definición de modelos

In [ ]:
# Diccionario que guarda el modelo de ML y los hiperparámetros a probar de cada modelo

models_params = {
  'LogisticRegression': {
    'model': LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced'),
    'params': {
      'C': [0.01, 0.1, 10, 100], 
      'solver': ['lbfgs']
    }
  },
  'RandomForest': {
    'model': RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    'params': {
      'n_estimators': [100, 200],
      'max_depth': [10, 20, None],
      'min_samples_split': [2, 5, 10],
      'min_samples_leaf': [1, 2, 4]
    }
  },
  'SVM_RBF': { 
    'model': SVC(probability=True, random_state=42, class_weight='balanced'),
    'params': {
      'C': [0.1, 1, 10],
      'gamma': ['scale', 'auto', 0.1],
      'penalty': ['rbf']
    }
  },
  'KNN': {
    'model': KNeighborsClassifier(),
    'params': {
      'n_neighbors': [3, 5, 7, 9, 11],
      'weights': ['uniform', 'distance'],
      'p': [1, 2]
    }
  },
  'GradientBoosting': {
    'model': GradientBoostingClassifier(random_state=42),
    'params': {
      'n_estimators': [100, 200],
      'learning_rate': [0.01, 0.05, 0.1],
      'max_depth': [3, 5, 7],
      'subsample': [0.8, 1.0]
    }
  },
  'CatBoost': {
    'model': CatBoostClassifier(verbose=0, random_state=42, class_weights='balanced'),
    'params': {
      'iterations': [100, 200],
      'depth': [4, 6, 8],
      'learning_rate': [0.01, 0.05, 0.1],
      'l2_leaf_reg': [1, 3, 5]
    }
  }
}

### Optimización de XGBoost con Optuna

In [ ]:
best_models = {}

def objective(trial):
  params = {
    'n_estimators': trial.suggest_int('n_estimators', 100, 400),    
    'max_depth': trial.suggest_int('max_depth', 3, 8),
    'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
    'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    'gamma': trial.suggest_float('gamma', 0, 3),
    'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
    'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
    'random_state': 42,
    'eval_metric': 'logloss',
    'use_label_encoder': False
  }
  model = XGBClassifier(**params)
  # Validación cruzada manual para AUC
  auc_scores = []
  for train_idx, val_idx in cv.split(X_sample, y_sample):
    X_tr, X_val = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_val = y_sample.iloc[train_idx], y_sample.iloc[val_idx]
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=15, verbose=False)
    y_proba = model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, y_proba))
  return np.mean(auc_scores)


print("\nOptimizando XGBoost con Optuna...")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)
best_xgb_params = study.best_params
print("Mejores hiperparámetros para XGBoost:", best_xgb_params)

# Entrenar XGBoost final con todos los datos
xgb_best = XGBClassifier(**best_xgb_params, random_state=42, eval_metric='logloss',use_label_enconder=False)
xgb_best.fit(X_train_resampled, y_train_resampled)
best_models['XGBoost'] = xgb_best

### Optimización de los otros modelos con RandomizedSearchCV

In [ ]:

# Optimización del resto de modelos con RandomizedSearchCV
for name, mp in models_params.items():
  print(f"\nOptimizando {name} con RandomizedSearchCV...")
  model = mp['model'] 
  param_dist = mp['params']
  random_search = RandomizedSearchCV(
    model, param_distributions=param_dist, n_iter=20,
    cv=cv, scoring='roc_auc', random_state=42, n_jobs=-1
  )
  random_search.fit(X_sample, y_sample)
  print(f"Mejores parámetros para {name}: {random_search.best_params_}")
  # Entrenar modelo final con todos los datos
  final_model = random_search.best_estimator_
  final_model.fit(X_train_resampled, y_train_resampled)
  best_models[name] = final_model
  

### Evaluación en test

In [ ]:
# Función para evaluar un modelo ya entrenado en test
def evaluate_model(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  try:
    y_proba = model.predict_proba(X_test)[:, 1]
  except AttributeError:
    y_proba = model.decision_function(X_test)
  metrics = {
    'Model': model_name,
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba)
  }
  return metrics

results = []
probas_dict = {}

for name, model in best_models.items():
  metrics,y_proba = evaluate_model(model, X_test_scaled, y_test, name)
  results.append(metrics)
  probas_dict[name] = y_proba

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('AUC', ascending=False)
print("\nResultados finales:\n", results_df.round(4))
results_df.to_csv('resultados_modelos.csv')

### Gráfico de barras comparativo

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
results_df[metrics_to_plot].plot(kind='bar', figsize=(14, 6), colormap='viridis')
plt.title('Comparación de métricas de los modelos')
plt.ylabel('Puntuación')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('comparacion_modelos.png')
plt.show()

### Matrices de confusión para cada modelo

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,10))
axes = axes.flatten()
for idx, (name, model) in enumerate(best_models.items()):
  y_pred = model.predict(X_test_scaled)
  cm = confusion_matrix(y_test, y_pred)
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
  axes[idx].set_title(f'{name}\nMatriz de confusión')
  axes[idx].set_xlabel('Predicho')
  axes[idx].set_ylabel('Real')

# Ocultar último subplot
if len(best_models) < 8:
  axes[-1].axis('off')
plt.tight_layout()
plt.savefig('matrices_confusion.png')
plt.show()

### Curva ROC para cada modelo

In [ ]:
plt.figure(figsize=(10,8))
for name, model in probas_dict.items():
  y_proba = model.predict_proba(X_test_scaled)[:, 1]
  fpr, tpr, _ = roc_curve(y_test, y_proba)
  auc = roc_auc_score(y_test, y_proba)
  plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curvas ROC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('curvas_roc.png')
plt.show()

## Seleción del mejor modelo

In [ ]:
results_df['Score'] = results_df['Recall'] + results_df['AUC']
best_model_name = results_df['Score'].idxmax()
best_model = best_models[best_model_name]
print(f"\nMejor modelo seleccionado: {best_model_name}")
print(f"Recall: {results_df.loc[best_model_name, 'Recall']:.4f}, AUC: {results_df.loc[best_model_name, 'AUC']:.4f}")

### XAI con SHAP

In [ ]:
print("\nGenerando explicaciones SHAP para el mejor modelo...")
X_test_shap = X_test_scaled.copy()

# Selección de XAI según modelo
if best_model_name in ['RandomForest', 'XGBoost', 'GradientBoosting', 'CatBoost']:
  # Modelos básados en árboles: TreeExplainer
  explainer = shap.TreeExplainer(best_model)
  shap_values = explainer.shap_values(X_test_shap)
  if isinstance(shap_values, list):
    shap_values = shap_values[1]

else:
  # Modelos lineales: KernelExplainer
  background = shap.sample(X_train_resampled, 100) # 100 instanciasde entrenamiento
  explainer = shap.KernelExplainer(best_model.predict_proba, background)
  shap_values = explainer.shap_values(X_test_shap, nsamples=100)
  if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Summary plot (beeswarm)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_shap, feature_names=X.columns, show=False)
plt.title(f'SHAP Summary Plot - {best_model_name}')
plt.tight_layout()
plt.savefig('shap_summary.png')
plt.show()

# Bar plot de importancia global
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_shap, feature_names=X.columns, plot_type="bar", show=False)
plt.title(f'SHAP Feature Importance - {best_model_name}')
plt.tight_layout()
plt.savefig('shap_importance.png')
plt.show()